In [1]:
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(dotenv_path=Path(".env"))
huggingface_token = os.environ.get("HUGGINGFACE_TOKEN")

if not huggingface_token:
    raise ValueError("HUGGINGFACE_TOKEN 이 .env 파일에 없습니다.")

login(token=huggingface_token)


/home/mobile/jaeyoung/all-in-one-llm-book/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import wandb

from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline,
    Trainer,
)
from transformers.integrations import WandbCallback
from trl import DataCollatorForCompletionOnlyLM
import evaluate

# 모델과 토크나이저 불러오기
model_name = "google/gemma-2b-it"
use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_cuda else torch.float32)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype,
)

if use_cuda:
    model = model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_name)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 164/164 [00:00<00:00, 2232.29it/s]


### GPU 사용 확인 메모

- `device_map="auto"`는 `accelerate`를 이용해 모델을 자동으로 적절한 장치에 배치할 때 주로 사용합니다.
- 현재 노트북은 단일 GPU 사용 여부를 직접 확인하기 쉽도록 `torch.cuda.is_available()`로 확인한 뒤 `model.to("cuda")`를 호출하는 방식으로 바꿨습니다.
- 아래 셀은 모델 파라미터가 실제로 어느 장치에 올라가 있는지와, GPU 메모리가 사용되고 있는지를 빠르게 확인하기 위한 점검용 셀입니다.


In [3]:
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    print("current device:", torch.cuda.current_device())

print("model device:", next(model.parameters()).device)

if torch.cuda.is_available():
    print("allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))


cuda available: True
device count: 2
device name: NVIDIA GeForce RTX 2080 Ti
bf16 supported: True
current device: 0
model device: cuda:0
allocated GB: 4.668
reserved GB: 4.74


In [4]:
import datasets
dataset = datasets.load_dataset("jaehy12/news3")
element = dataset["train"][1]
element

{'original': ' “동선 봐도 도움이 안 되는 것 같아요.\n  어딘지 알아야 피해가고 조심할 텐데요.\n ”“확진자가 다녀간 곳은 방역 마쳤는데 동선 공개는 그 부근 일대를 싹 다 죽이자는 것 같아요.\n ” 최근 부천시 페이스북 신종 코로나바이러스 감염증(코로나19) 관련 게시물에는 확진자 동선 공개 범위를 성토하는 댓글이 여럿 달렸다.\n  물류센터·교회 발 코로나19 확산이 지역사회 N차 감염으로 이어지면서 확진자 동선을 확대 공개해야 한다는 주장이 제기된 가운데 사생활 침해 등을 근거로 반대하는 의견도 나오면서 동선공개 논란에 다시 불이 붙는 모양새다.\n  지난 2월 코로나19 초기엔 확진자 동선이 대부분 공개됐다.\n  쇼핑몰을 방문한 확진자가 시간대별로 어느 매장을 찾았는지 등의 동선이 지자체 소셜네트워크서비스(SNS)에 게재됐다.\n  이후 사생활 침해 논란이 일자 질본은 감염병 예방에 필요한 정보에 한해 확진자 정보를 공개하라는 권고사항을 발표했다.\n  권고사항에는 증상 발생 2일 전부터 격리일까지의 시간, 감염을 우려할 만큼의 확진자와의 접촉이 일어난 장소 및 수단 등을 공개한다는 내용이 담겼다.\n  해당 공간 내 모든 접촉자가 파악되면 공개하지 않을 수 있다는 단서도 추가됐다.\n  확진자가 마지막 접촉자와 만날 일로부터 14일이 지나면 공개한 동선을 삭제할 것도 권고했다.\n  확진 늘자 동선공개 확대 나선 지자체 물류센터·교회 등 집단감염에 이어 감염경로가 특정되지 않은 확진 사례가 늘면서 일부 지자체는 동선 공개 범위를 확대하고 있다.\n  김포시는 지난 2일부터 확진자 이동 경로에 따른 동선 공개 원칙을 제한적으로 확대했다.\n  시민 우려 불식을 위해 확진자 방문으로 접촉자가 다수 발생하고 확산이 우려되는 장소는 상호를 공개하기로 한 것이다.\n  앞서 부천 118번 확진자인 A씨(31)가 한 제약회사 영업사원인 사실이 알려지면서 SNS에 그가 다닌 병원 정보가 공유됐다.\n  장덕천 부천시장은 “SNS에 A씨가 평소